In [ ]:
import torch
import torchaudio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import scipy.signal
import random
from IPython.display import Audio
import os

# Asegurar reproducibilidad
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)

## Descargar datos necesarios

In [ ]:
import urllib.request
import zipfile

# Descargar ruidos MUSAN
url = 'http://dihana.cps.unizar.es/~cadrete/curso_asr_unizar/musan_small.zip'
if not os.path.exists('musan_small.zip'):
    print("Descargando musan_small.zip...")
    urllib.request.urlretrieve(url, 'musan_small.zip')
    with zipfile.ZipFile('musan_small.zip', 'r') as zip_ref:
        zip_ref.extractall()
    print("Descargado y descomprimido musan_small")

# Descargar RIRs (Room Impulse Responses)
url = 'http://dihana.cps.unizar.es/~cadrete/curso_asr_unizar/rirs_noises_small.zip'
if not os.path.exists('rirs_noises_small.zip'):
    print("Descargando rirs_noises_small.zip...")
    urllib.request.urlretrieve(url, 'rirs_noises_small.zip')
    with zipfile.ZipFile('rirs_noises_small.zip', 'r') as zip_ref:
        zip_ref.extractall()
    print("Descargado y descomprimido rirs_noises_small")

# Descargar datos de fechas2
url = 'https://dihana.unizar.es/~cadrete/valencia2526/fechas2.zip'
if not os.path.exists('../fechas2.zip'):
    print("Descargando fechas2.zip...")
    urllib.request.urlretrieve(url, '../fechas2.zip')
    with zipfile.ZipFile('../fechas2.zip', 'r') as zip_ref:
        zip_ref.extractall('..')
    print("Descargado y descomprimido fechas2")

## Clases de Data Augmentation

In [ ]:
class AdditiveNoise:
    """Añade ruido aditivo con una SNR específica."""
    
    def __init__(self, noise_files, snr_db=10, sample_rate=16000):
        """
        Args:
            noise_files: Lista de archivos de ruido
            snr_db: Relación señal-ruido en dB
            sample_rate: Frecuencia de muestreo
        """
        self.noise_files = noise_files
        self.snr_db = snr_db
        self.sample_rate = sample_rate
    
    def __call__(self, audio):
        """Aplica ruido aditivo al audio.
        
        Args:
            audio: Tensor de audio (1D)
        
        Returns:
            Audio con ruido añadido
        """
        if len(self.noise_files) == 0:
            return audio
        
        # Seleccionar archivo de ruido aleatorio
        noise_file = random.choice(self.noise_files)
        
        # Cargar ruido
        noise, fs = torchaudio.load(noise_file)
        noise = noise[0]  # Tomar primer canal
        
        # Verificar frecuencia de muestreo
        if fs != self.sample_rate:
            resampler = torchaudio.transforms.Resample(fs, self.sample_rate)
            noise = resampler(noise)
        
        # Extraer segmento de ruido de la misma longitud que el audio
        if len(noise) < len(audio):
            # Si el ruido es más corto, repetirlo
            repeats = (len(audio) // len(noise)) + 1
            noise = noise.repeat(repeats)
        
        # Seleccionar segmento aleatorio
        start = random.randint(0, len(noise) - len(audio))
        noise = noise[start:start + len(audio)]
        
        # Calcular potencias
        p_audio = audio.std() ** 2
        p_noise = noise.std() ** 2
        
        # Ajustar potencia del ruido según SNR deseada
        noise = noise * torch.sqrt(p_audio / p_noise) * torch.pow(torch.tensor(10.0), -self.snr_db / 20)
        
        # Añadir ruido
        return audio + noise


class ReverbAugmentation:
    """Añade reverberación usando Room Impulse Responses (RIR)."""
    
    def __init__(self, rir_files, sample_rate=16000):
        """
        Args:
            rir_files: Lista de archivos RIR
            sample_rate: Frecuencia de muestreo
        """
        self.rir_files = rir_files
        self.sample_rate = sample_rate
    
    def __call__(self, audio):
        """Aplica reverberación al audio.
        
        Args:
            audio: Tensor de audio (1D)
        
        Returns:
            Audio con reverberación
        """
        if len(self.rir_files) == 0:
            return audio
        
        # Seleccionar RIR aleatorio
        rir_file = random.choice(self.rir_files)
        
        # Cargar RIR
        rir, fs = torchaudio.load(rir_file)
        rir = rir[0].numpy()  # Tomar primer canal
        
        # Normalizar RIR
        rir = rir / np.max(np.abs(rir))
        
        # Aplicar convolución
        audio_reverb = scipy.signal.convolve(audio.numpy(), rir, mode='full')
        
        # Recortar al tamaño original
        audio_reverb = audio_reverb[:len(audio)]
        
        return torch.tensor(audio_reverb, dtype=torch.float32)

## Cargar archivos de ruido y RIR

In [ ]:
# Cargar archivos de ruido
noise_files = glob.glob('musan_small/**/*.wav', recursive=True)
print(f"Archivos de ruido encontrados: {len(noise_files)}")

# Cargar archivos RIR
rir_files = glob.glob('RIRS_NOISES_small/**/*.wav', recursive=True)
print(f"Archivos RIR encontrados: {len(rir_files)}")

## Dataset para Fechas2

In [ ]:
class Fechas2Dataset(torch.utils.data.Dataset):
    """Dataset para la tarea de fechas2."""
    
    def __init__(self, csv_file, audio_len=4*16000, transforms=None, sample_rate=16000):
        """
        Args:
            csv_file: Archivo CSV con columnas 'wav' y 'txt'
            audio_len: Longitud máxima del audio en samples
            transforms: Lista de transformaciones a aplicar
            sample_rate: Frecuencia de muestreo
        """
        self.df = pd.read_csv(csv_file)
        self.audio_len = audio_len
        self.transforms = transforms if transforms is not None else []
        self.sample_rate = sample_rate
        
        # Construir rutas completas
        self.csv_dir = os.path.dirname(csv_file)
        
        print(f"Dataset cargado: {len(self.df)} ejemplos")
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        """Obtiene un elemento del dataset.
        
        Returns:
            audio: Tensor de audio
            text: Texto transcrito
        """
        row = self.df.iloc[idx]
        
        # Construir ruta del archivo de audio
        audio_path = os.path.join(self.csv_dir, row['wav'])
        if not os.path.exists(audio_path):
            audio_path = os.path.join('..', row['wav'])
        
        # Cargar audio
        audio, fs = torchaudio.load(audio_path)
        audio = audio[0]  # Tomar primer canal
        
        # Resamplear si es necesario
        if fs != self.sample_rate:
            resampler = torchaudio.transforms.Resample(fs, self.sample_rate)
            audio = resampler(audio)
        
        # Padding o recorte
        if len(audio) < self.audio_len:
            audio = torch.nn.functional.pad(audio, (0, self.audio_len - len(audio)), value=0)
        else:
            audio = audio[:self.audio_len]
        
        # Aplicar transformaciones (data augmentation)
        for transform in self.transforms:
            audio = transform(audio)
        
        text = row['txt']
        
        return audio, text


class Fechas2TestDataset(torch.utils.data.Dataset):
    """Dataset de test sin augmentation."""
    
    def __init__(self, csv_file, audio_len=4*16000, sample_rate=16000):
        self.df = pd.read_csv(csv_file)
        self.audio_len = audio_len
        self.sample_rate = sample_rate
        self.csv_dir = os.path.dirname(csv_file)
        
        print(f"Test dataset cargado: {len(self.df)} ejemplos")
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        audio_path = os.path.join(self.csv_dir, row['wav'])
        if not os.path.exists(audio_path):
            audio_path = os.path.join('..', row['wav'])
        
        audio, fs = torchaudio.load(audio_path)
        audio = audio[0]
        
        if fs != self.sample_rate:
            resampler = torchaudio.transforms.Resample(fs, self.sample_rate)
            audio = resampler(audio)
        
        if len(audio) < self.audio_len:
            audio = torch.nn.functional.pad(audio, (0, self.audio_len - len(audio)), value=0)
        else:
            audio = audio[:self.audio_len]
        
        text = row['txt']
        
        return audio, text

## Crear datasets con augmentation

In [ ]:
# Crear transformaciones
noise_aug = AdditiveNoise(noise_files, snr_db=10)
reverb_aug = ReverbAugmentation(rir_files)

# Dataset de entrenamiento con augmentation
trainset = Fechas2Dataset(
    '../fechas2/fechas2_train.es.csv',
    audio_len=4*16000,
    transforms=[noise_aug, reverb_aug]
)

# Dataset de test sin augmentation
testset = Fechas2TestDataset(
    '../fechas2/fechas2_test.es.csv',
    audio_len=4*16000
)

## Visualizar ejemplos

In [ ]:
# Obtener un ejemplo
audio, text = trainset[0]
print(f"Forma del audio: {audio.shape}")
print(f"Texto: {text}")
print(f"Duración: {len(audio) / 16000:.2f} segundos")

## Comparar audio original vs con augmentation

In [ ]:
# Cargar audio original sin augmentation
testset_no_aug = Fechas2TestDataset('../fechas2/fechas2_train.es.csv')
audio_orig, text = testset_no_aug[0]

# Cargar mismo audio con augmentation
audio_aug, _ = trainset[0]

print(f"Texto: {text}")
print(f"\nAudio original:")
display(Audio(audio_orig.numpy(), rate=16000))

print(f"\nAudio con augmentation (ruido + reverberación):")
display(Audio(audio_aug.numpy(), rate=16000))

## Visualizar espectrogramas

In [ ]:
def plot_spectrogram(audio, title="Espectrograma"):
    """Plotea el espectrograma de un audio."""
    spectrogram_transform = torchaudio.transforms.Spectrogram(
        n_fft=512,
        win_length=400,  # 25ms a 16kHz
        hop_length=160   # 10ms a 16kHz
    )
    
    spec = spectrogram_transform(audio)
    spec_db = 10 * torch.log10(spec + 1e-10)
    
    plt.imshow(spec_db.numpy(), aspect='auto', origin='lower', cmap='jet')
    plt.colorbar(format='%+2.0f dB')
    plt.title(title)
    plt.xlabel('Frames')
    plt.ylabel('Frecuencia')

# Comparar espectrogramas
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plot_spectrogram(audio_orig, "Audio Original")

plt.subplot(1, 2, 2)
plot_spectrogram(audio_aug, "Audio con Augmentation")

plt.tight_layout()
plt.show()

## Visualizar múltiples ejemplos

In [ ]:
# Visualizar varios ejemplos del dataset
n_examples = 5

fig, axes = plt.subplots(n_examples, 2, figsize=(15, 3*n_examples))

for i in range(n_examples):
    # Audio original
    audio_orig, text = testset_no_aug[i]
    
    # Audio con augmentation
    audio_aug, _ = trainset[i]
    
    # Calcular espectrogramas
    spec_transform = torchaudio.transforms.Spectrogram(
        n_fft=512, win_length=400, hop_length=160
    )
    
    spec_orig = 10 * torch.log10(spec_transform(audio_orig) + 1e-10)
    spec_aug = 10 * torch.log10(spec_transform(audio_aug) + 1e-10)
    
    # Plotear
    axes[i, 0].imshow(spec_orig.numpy(), aspect='auto', origin='lower', cmap='jet')
    axes[i, 0].set_title(f"Original: '{text}'")
    axes[i, 0].set_ylabel('Frecuencia')
    
    axes[i, 1].imshow(spec_aug.numpy(), aspect='auto', origin='lower', cmap='jet')
    axes[i, 1].set_title(f"Con Augmentation: '{text}'")
    axes[i, 1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

print("\nEjemplos de audio con y sin augmentation:")
for i in range(min(3, n_examples)):
    audio_orig, text = testset_no_aug[i]
    audio_aug, _ = trainset[i]
    
    print(f"\n--- Ejemplo {i+1}: '{text}' ---")
    print("Original:")
    display(Audio(audio_orig.numpy(), rate=16000))
    print("Con augmentation:")
    display(Audio(audio_aug.numpy(), rate=16000))

## Estadísticas del dataset

In [ ]:
print(f"Dataset de entrenamiento: {len(trainset)} ejemplos")
print(f"Dataset de test: {len(testset)} ejemplos")

# Analizar longitud de textos
train_df = pd.read_csv('../fechas2/fechas2_train.es.csv')
test_df = pd.read_csv('../fechas2/fechas2_test.es.csv')

train_lengths = [len(text.split()) for text in train_df['txt']]
test_lengths = [len(text.split()) for text in test_df['txt']]

print(f"\nLongitud promedio de texto (train): {np.mean(train_lengths):.2f} palabras")
print(f"Longitud promedio de texto (test): {np.mean(test_lengths):.2f} palabras")
print(f"Longitud máxima de texto (train): {np.max(train_lengths)} palabras")
print(f"Longitud máxima de texto (test): {np.max(test_lengths)} palabras")

## Crear DataLoaders

In [ ]:
# Crear dataloaders
batch_size = 16

# Nota: Para usar DataLoader con textos de longitud variable,
# necesitamos una función collate personalizada
def collate_fn(batch):
    """Función para agrupar batch con textos de longitud variable."""
    audios, texts = zip(*batch)
    
    # Apilar audios (todos tienen la misma longitud por padding)
    audios = torch.stack(audios)
    
    return audios, list(texts)

train_loader = torch.utils.data.DataLoader(
    trainset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
)

test_loader = torch.utils.data.DataLoader(
    testset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn
)

print(f"Train loader: {len(train_loader)} batches")
print(f"Test loader: {len(test_loader)} batches")

# Probar un batch
audios_batch, texts_batch = next(iter(train_loader))
print(f"\nForma del batch de audio: {audios_batch.shape}")
print(f"Número de textos: {len(texts_batch)}")
print(f"Primeros 3 textos: {texts_batch[:3]}")

## Guardar configuración

In [ ]:
# Guardar información del dataset para uso posterior
import pickle

dataset_config = {
    'audio_len': 4*16000,
    'sample_rate': 16000,
    'train_size': len(trainset),
    'test_size': len(testset),
    'batch_size': batch_size,
    'num_noise_files': len(noise_files),
    'num_rir_files': len(rir_files),
}

with open('dataset_config.pkl', 'wb') as f:
    pickle.dump(dataset_config, f)

print("Configuración del dataset guardada en 'dataset_config.pkl'")